# TabICL Oracle — One-cell Colab runner

**Before running:** Runtime → Change runtime type → **T4 GPU** (or A100/V100 if Pro).

**Required secret:** Click 🔑 in left sidebar → add `HF_TOKEN` (with `LBJLincoln26` write access) → toggle "Notebook access" on.

Then click **Runtime → Run all**. That's it.

In [ ]:
# ============================================================================
# Self-contained Colab runner for TabICL NBA oracle training.
# Pulls the LATEST trainer from public HF dataset; no git, no manual setup.
# REQUIRE_GPU=1 means hard-fail if no CUDA detected (catches silent CPU runs).
# ============================================================================
import os, subprocess, sys

# 1. GPU sanity
subprocess.run(['nvidia-smi', '-L'], check=False)
import torch
assert torch.cuda.is_available(), \
    'No CUDA. Runtime → Change runtime type → T4 GPU → Save → Connect → re-run.'
print(f'✓ GPU: {torch.cuda.get_device_name(0)}')

# 2. HF token from Colab secrets (one-time setup in left sidebar key icon)
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'Add HF_TOKEN to Colab secrets first (left sidebar 🔑)'
print(f'✓ HF_TOKEN set (len={len(os.environ["HF_TOKEN"])})')

# 3. Deps
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'tabicl', 'xgboost', 'lightgbm', 'huggingface_hub'], check=True)

# 4. Pull latest trainer (always-fresh from public HF dataset — bypasses any
#    stale notebook code the user might have copied days ago).
subprocess.run([
    'wget', '-qO', 'train_tabicl_oracle.py',
    'https://huggingface.co/datasets/LBJLincoln26/nba-oracle-model/resolve/main/train_tabicl_oracle.py'
], check=True)
print(f'✓ Trainer downloaded ({os.path.getsize("train_tabicl_oracle.py")} bytes)')

# 5. Run with REQUIRE_GPU=1 so it fails-fast instead of falling back to CPU
os.environ['REQUIRE_GPU'] = '1'
subprocess.run([sys.executable, 'train_tabicl_oracle.py'], check=True)